# Hidden CKD

#### Features
The following are the features used in this study in order of appearance
- Date of event: Date of the screening
- Gender: Gender of the patient (M: Male, F: Female)
- Ethnicity: The ethnicity of the participant
- Age: Age of the patient (years)
- Height (cm): Height of the participant in cm
- Weight (kg): Weight of the participant in kg
- BMI: BMI of the participant
- BMI Category: Classification of the particpant BMI according to NICE guidelines
- Systolic, Diastolic: The systolic and diastolic of the partcipants
- Pulse Pressure: 
- BP Category: Classification of the particpant BP according to NICE guidelines
- Has High BP:
- Has Diabetes
- Has Kidney Disease
- On BP Medication?
- On Diabetes Medication?
- On Cholesterol Medication?
- On Other Medication?
- Family History of Kidney Disease: Whether the patient has a family history of kidney disease (Definitely Yes, Definitely Not, Not Sure)
- uACR: Urine albumin to creatinine ratio of the participants as measured using a urine dipstick (Normal, Abnormal, High Abnormal)
- CKD Risk: A calculation of CKD risk by combining research findings

This is how CKD risk is calculated

low risk = sys<140, dia<90, uACR='Normal', Has_Diabetes=False or Family_KD=False<br>
moderate risk = sys<140, dia<90, uACR='Abnormal', Has_Diabetes=True or Family_KD=True<br>
high risk = sys>=180, dia>=120, uACR='High Abnormal', Has_Diabetes=False or Family_KD=False<br>

In [8]:
import numpy as np
import pandas as pd
from src.config import PROCESSED_DATA_DIR

In [9]:
filename = PROCESSED_DATA_DIR / 'hidden_ckd_processed.csv'
data = pd.read_csv(filename)
data.head()

,Date,Gender,Ethnicity,S_Ethnicity,Ethnicity_Black,DOB,Age,Age_Category,Height,Weight,...,Has_Diabetes,Has_KD,Has_HD,BP_Meds,Diabetes_Meds,Cholesterol_Meds,Other_Meds,Family_KD,uACR,CKD_Risk
0,23/10/2022,Male,Black Caribbean,Black,True,21/05/1946,76.5,>70,161.0,64.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
1,23/10/2022,Male,Black African (West Africa),Black,True,25/01/1970,52.8,41-55,163.0,78.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
2,26/08/2023,Male,Black Caribbean,Black,True,14/07/2005,18.1,<25,167.0,91.0,...,False,False,False,False,False,False,False,Definitely not,Normal,Low
3,28/04/2023,Male,Black Caribbean,Black,True,25/04/1969,54.0,41-55,168.0,87.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
4,06/11/2022,Female,Black African (West Africa),Black,True,03/11/1979,43.0,41-55,187.0,109.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate


In [10]:
data.groupby(by=['Has_Hpt', 'Ethnicity_Black'])[['Age', 'Height', 'Weight', 'Systolic', 'Diastolic', 'Pulse_Pressure', 'MAP']].median()

Age  Height  Weight  Systolic  Diastolic  \
Has_Hpt Ethnicity_Black                                               
False   False            46.85   167.0    81.8     128.5       80.0   
        True             50.15   165.0    80.0     130.0       80.0   
True    False            58.20   162.0    86.3     146.5       88.5   
        True             58.30   164.0    81.0     145.0       87.0   

                         Pulse_Pressure     MAP  
Has_Hpt Ethnicity_Black                          
False   False                      49.0   96.85  
        True                       50.0   96.70  
True    False                      58.5  108.35  
        True                       59.0  107.30

# **Exploratory Data Analysis**

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [12]:
data.describe()

,Age,Height,Weight,BMI,Systolic,Diastolic,Pulse_Pressure,MAP
count,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000
mean,50.901457,165.813309,81.799761,29.841207,136.210198,82.668054,53.542144,100.514984
std,14.165017,9.140738,15.422907,5.720421,20.212144,11.863947,15.003352,13.415813
min,14.400000,111.500000,45.500000,19.000000,88.000000,45.000000,11.000000,65.000000
25%,42.100000,159.000000,71.000000,25.800000,123.000000,75.000000,44.000000,91.300000
50%,52.900000,165.000000,80.200000,29.200000,134.000000,82.000000,52.000000,98.700000
75%,60.300000,171.000000,90.000000,32.900000,146.000000,90.000000,61.000000,108.300000
max,92.100000,198.000000,169.000000,62.100000,243.000000,141.000000,154.000000,164.000000


In [13]:
# Pairwise Pearson correlations between numeric variables
cr = data[['Age', 'Height', 'Weight', 'BMI', 'Systolic', 'Diastolic', 'Pulse_Pressure', 'MAP']].corr(method='pearson')

fig = go.Figure(go.Heatmap(
    x=cr.columns,
    y=cr.columns,
    z=cr.values.tolist(),
    colorscale='RdBu', zmin=-1, zmax=1
))

fig.update_layout(
    autosize=False,
    width=1100,
    height=400,
    margin=dict(l=20, r=20, t=50, b=20),
    bargap=0.15,
    title=dict(text='Pearson Correlations')
)

fig.show()

Systolic and diastolic BP readings are weakly positively correlated with age and weight as expected

## **Grouped/Aggregated Statistics**

In [7]:
from src.utils.misc_utils import group_stats

In [8]:
eth_data = group_stats(data, 'Ethnicity')
eth_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
Ethnicity,,,,,,,,,,,,,,
Any other,53.28,170.88,97.35,33.10,133.25,82.25,99.25,4,75.00,25.00,0.00,25.00,75.00,0.00
Asian other,50.09,168.34,84.01,29.66,128.96,78.87,95.57,23,65.22,30.43,4.35,30.43,65.22,4.35
Bangladeshi,59.20,167.91,80.63,28.44,152.71,84.71,107.36,7,85.71,14.29,0.00,0.00,100.00,0.00
Black African (Central Africa),51.42,168.65,83.70,29.49,138.00,83.93,101.95,15,66.67,33.33,0.00,13.33,86.67,0.00
Black African (East Africa),47.48,163.08,78.08,29.44,128.24,79.24,95.57,21,47.62,42.86,9.52,28.57,61.90,9.52
Black African (North Africa),54.88,170.75,77.33,26.58,130.00,80.83,97.23,6,83.33,16.67,0.00,33.33,66.67,0.00
Black African (South Africa),46.72,164.92,82.48,30.28,126.00,78.50,94.33,6,16.67,83.33,0.00,0.00,100.00,0.00
Black African (West Africa),50.98,166.03,81.88,29.79,136.51,82.84,100.72,499,52.91,39.68,7.41,28.66,61.32,10.02
Black African (unspecified),51.32,164.79,81.33,30.13,136.57,83.36,101.10,149,51.68,41.61,6.71,28.19,58.39,13.42


The vast majority of the project participants are Black. It would not be helpful to attempt to make inferences based on 'Ethnicity' given how unbalanced it is.

In [9]:
s_eth_data = group_stats(data, 'S_Ethnicity')
s_eth_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
S_Ethnicity,,,,,,,,,,,,,,
Asian other,50.09,168.34,84.01,29.66,128.96,78.87,95.57,23,65.22,30.43,4.35,30.43,65.22,4.35
Black,50.88,165.83,81.55,29.75,136.17,82.66,100.50,817,52.63,41.00,6.36,27.78,62.79,9.42
Mixed,56.42,163.27,82.89,31.05,140.85,82.67,102.06,33,54.55,36.36,9.09,21.21,69.70,9.09
Other,53.28,170.88,97.35,33.10,133.25,82.25,99.25,4,75.00,25.00,0.00,25.00,75.00,0.00
South Asian,52.01,168.48,82.23,29.08,141.31,84.91,103.70,32,78.12,21.88,0.00,15.62,81.25,3.12
White,47.23,164.03,82.64,30.78,134.21,83.06,100.11,52,75.00,21.15,3.85,38.46,50.00,11.54


This table begins to open up some key insights. It appears that participants who are Black or are of Black descent are at the highest risk of high abnormal uACR readings, which is consistent with literature (a quick peak at the breakdown for the 'Mixed' Simplified Ethnicity shows that only Mixed people of Black descent have high abnormal uACR readings). None of the participants classed 'Indian' (i.e. Indian, Pakistani, and Bangladeshi) have high abnormal uACR readings. Across the board, it appears Black people have a higher risk of abnormal and high abnormal uACR even if they do not have the highest BP readings, which is a risk factor for abnormal uACR values.

In [10]:
eth_black_data = group_stats(data, 'Ethnicity_Black')
eth_black_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
Ethnicity_Black,,,,,,,,,,,,,,
False,49.79,165.99,83.38,30.31,135.52,82.78,100.36,118,73.73,23.73,2.54,28.81,64.41,6.78
True,51.06,165.79,81.58,29.78,136.31,82.65,100.54,843,52.55,40.93,6.52,27.64,62.87,9.49


This table further drives home the point that Black people are at increased risk of abnormal uACR readings.

In [11]:
gender_data = group_stats(data, 'Gender')
gender_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
Gender,,,,,,,,,,,,,,
Female,51.20,165.57,81.31,29.74,136.80,82.57,100.64,540,51.11,43.89,5.00,24.81,67.04,8.15
Male,50.52,166.12,82.43,29.97,135.46,82.80,100.35,421,60.33,32.30,7.36,31.59,57.96,10.45


This table shows that for comparable BMI and BP readings, females are at moderately increased risk of having abnormal uACR readings. This could be due to undertesting on the part of males or due to the naturally increased protein levels in the urine of females.

In [12]:
age_data = group_stats(data, 'Age_Category').iloc[[3,0,1,2,4], :]
age_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
Age_Category,,,,,,,,,,,,,,
<25,21.80,168.81,74.43,26.11,126.37,75.19,92.25,52,71.15,28.85,0.00,51.92,46.15,1.92
25-40,33.32,167.09,79.95,28.70,127.67,80.11,95.97,157,63.69,28.66,7.64,44.59,47.77,7.64
41-55,48.45,166.07,84.44,30.67,136.43,83.81,101.35,354,55.93,39.83,4.24,27.12,64.41,8.47
56-70,61.02,165.06,81.79,30.15,139.54,84.20,102.65,327,49.54,43.73,6.73,20.18,69.72,10.09
>70,76.71,162.97,78.17,29.56,145.89,81.03,102.65,71,46.48,40.85,12.68,11.27,71.83,16.90


Age is quite clearly an important risk factor for abnormal uACR readings (as BP readings also increase)

In [13]:
bp_cat_data = group_stats(data, 'BP_Category').iloc[[3,4,0,1,2], :]
bp_cat_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
BP_Category,,,,,,,,,,,,,,
NORMAL,48.26,165.70,79.84,29.16,123.15,75.95,91.68,540,58.89,36.48,4.63,49.44,45.93,4.63
PRE HPT,53.67,165.82,83.43,30.49,144.04,88.72,107.16,156,48.72,44.87,6.41,0.00,92.31,7.69
HPT 1,53.57,165.79,84.34,30.81,151.25,90.05,110.45,197,55.84,37.06,7.11,0.00,89.85,10.15
HPT 2,58.33,167.53,86.83,30.94,174.63,98.81,124.08,59,38.98,47.46,13.56,0.00,62.71,37.29
HPT CRISIS,54.60,161.84,82.59,31.43,202.78,113.67,143.36,9,33.33,55.56,11.11,0.00,0.00,100.00


High BP is a key contributor to abnormal uACR readings.

In [14]:
fam_data = group_stats(data, 'Family_KD')
fam_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
Family_KD,,,,,,,,,,,,,,
Definitely not,50.55,165.40,81.09,29.72,134.60,82.09,99.59,511,51.47,42.86,5.68,26.22,65.95,7.83
Definitely yes,50.80,166.41,80.62,29.09,135.81,81.42,99.55,36,61.11,30.56,8.33,33.33,55.56,11.11
Not sure,51.35,166.27,82.78,30.06,138.23,83.50,101.74,414,59.18,34.54,6.28,29.23,60.14,10.63


There is a moderate increase in abnormal uACR readings in participants who definitely have a family history of kidney disease compared with those who do not. Readings are abit of a mixed bag for participants who are not sure. With that said, literature indicates that a troubling proportion of CKD cases in the UK go undiagnosed.

In [15]:
bmi_data = group_stats(data, 'BMI_Category').iloc[[0,2,1], :]
bmi_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
BMI_Category,,,,,,,,,,,,,,
HEALTHY,46.97,169.20,66.08,23.01,129.62,78.53,95.56,184,63.59,31.52,4.89,39.13,53.26,7.61
OVERWEIGHT,51.12,166.88,76.98,27.57,135.69,82.35,100.13,361,57.06,38.50,4.43,29.09,64.82,6.09
OBESE,52.45,163.39,92.94,34.83,139.57,84.78,103.04,416,49.76,42.31,7.93,21.63,65.87,12.50


BMI is an important factor in determining one's risk of abnormal uACR readings.

In [16]:
hkd_data = group_stats(data, 'Has_KD')
hkd_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
Has_KD,,,,,,,,,,,,,,
False,50.83,165.82,81.86,29.86,136.16,82.65,100.49,945,55.77,38.52,5.71,28.15,62.96,8.89
True,55.22,165.35,78.48,28.86,139.25,83.75,102.26,16,18.75,56.25,25.00,6.25,68.75,25.00


Obviously, participants with kidney disease are far more likely to have abnormal uACR readings than those without it.

In [17]:
hdiabetes_data = group_stats(data, 'Has_Diabetes')
hdiabetes_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
Has_Diabetes,,,,,,,,,,,,,,
False,49.93,166.26,82.02,29.77,135.83,82.84,100.51,863,56.78,38.01,5.21,30.94,60.83,8.23
True,59.46,161.89,79.85,30.46,139.55,81.11,100.59,98,40.82,45.92,13.27,0.00,82.65,17.35


In [18]:
hdiabetes_data = group_stats(data[data['BP_Meds']==False], 'Has_Diabetes')
hdiabetes_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
Has_Diabetes,,,,,,,,,,,,,,
False,48.51,166.48,81.65,29.56,133.16,81.66,98.82,702,59.12,36.61,4.27,37.75,55.70,6.55
True,56.33,163.16,80.13,30.21,132.02,79.20,96.80,46,52.17,30.43,17.39,0.00,78.26,21.74


In [19]:
data.groupby(by=['Has_Hpt', 'Cholesterol_Meds'])[['Age', 'BMI', 'Systolic', 'Diastolic', 'MAP']].mean()

Age        BMI    Systolic  Diastolic  \
Has_Hpt Cholesterol_Meds                                                
False   False             48.553531  29.529096  132.483051  81.240113   
        True              58.756250  31.381250  133.312500  80.375000   
True    False             57.322685  30.795833  147.486111  87.189815   
        True              58.028571  29.371429  148.095238  86.047619   

                                 MAP  
Has_Hpt Cholesterol_Meds              
False   False              98.320339  
        True               98.025000  
True    False             107.288426  
        True              106.733333

Diabetes is an important risk factor in the development and progression of kidney disease. This is as a result of the gradual damage diabetes causes to the kidneys over time.

In [20]:
bpmeds_data = group_stats(data, 'BP_Meds')
bpmeds_data

,Mean Age,Mean Height (cm),Mean Weight (kg),Mean BMI,Mean Systolic,Mean Diastolic,Mean MAP,Count,Normal uACR %,Abnormal uACR %,High Abnormal uACR %,Low Risk %,Moderate Risk %,High Risk %
BP_Meds,,,,,,,,,,,,,,
False,48.99,166.27,81.55,29.60,133.09,81.50,98.7,748,58.69,36.23,5.08,35.43,57.09,7.49
True,57.60,164.19,82.67,30.68,147.18,86.76,106.9,213,42.72,47.89,9.39,0.94,84.04,15.02


## **Data Visualisations**

In [ ]:
from src.utils.misc_utils import multi_stop_gradient, create_cmap

# Create a custom colourmap
colours = ["#FF3A00", "#FAA272", "#F3C389", "#0088ff", "#35488A"]
alt_colours = ['#F3C389', '#35488A', '#FF3A00', '#D55B64', '#B23613', '#003376']
cmap = create_cmap(colours)

In [23]:
hist_features = ['Age', 'Height', 'Weight', 'BMI', 'Systolic', 'Diastolic']
colour = '#35488A'

# Initialize subplots
fig = make_subplots(rows=2, cols=1, row_heights=[0.2, 0.8], shared_xaxes=True)

# Add Traces with visibility settings
for idx, feature in enumerate(hist_features):
    fig.add_trace(go.Histogram(
        x=data[feature],
        nbinsx=len(np.histogram_bin_edges(data[feature], bins='fd')),
        name=feature,
        marker_color=colour,
        visible=(idx == 0)
    ), row=2, col=1)
    fig.add_trace(go.Box(
        x=data[feature],
        marker_symbol='line-ns-open',
        boxpoints='all',
        jitter=0,
        hoveron='points',
        name=feature,
        marker_color=colour,
        visible=(idx == 0)
    ), row=1, col=1)


# Add buttons
fig.update_layout(
    updatemenus=[
        dict(
            direction="down",
            showactive=True,
            x=-0.2,
            xanchor='left',
            y=0.9,
            yanchor='top',
            buttons=list([
                dict(label=feature,
                     method="update",
                     args=[{"visible": [(i // 2 == idx) for i in range(len(hist_features) * 2)]},
                          {"xaxis2.title": feature}])
                for idx, feature in enumerate(hist_features)
            ]),
        )
    ],
    showlegend=False,
    title=dict(text='Histograms', x=0.01),
    yaxis1_title="",
    xaxis2_title=hist_features[0], # Default x-axis title for the first histogram
    yaxis2_title='Count', # y-axis title
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="Feature:", showarrow=False,
        x = -0.2, xref="paper", y=1, yref="paper", align="left")
    ]
)

# Set plot size and add a bar gap
fig.update_layout(
    autosize=False,
    width=1100,
    height=400,
    margin=dict(l=20, r=20, t=50, b=20),
    bargap=0.15
)


fig.show()

Height, Weight, and BMI data are skewed.

In [24]:
# Variables
k_vars = ['BMI_Category', 'BP_Category', 'S_Ethnicity', 'Ethnicity_Black', 'Gender']
x_vars = ['uACR', 'BP_Category', 'S_Ethnicity']
y_var = 'Age'
colour_list = ['DarkOrange', 'Sienna', 'Chocolate', 'DarkSalmon', 'Coral', 'SandyBrown']

def barplots(df, k_vars, x_vars, y_var, title):
    # Initial x_var
    current_x_var = x_vars[0]

    fig = go.Figure()

    # Add traces for each combination of k_var values
    for a in k_vars:
        for col, k in enumerate(df[a].unique()):
            group_values = sorted(df[group_by].unique())
            n_groups = len(group_values)
            colours = multi_stop_gradient(colours, n_groups)
            colour_map = dict(zip(group_values, colours))
            fig.add_trace(
                go.Bar(
                    name=f'{k}',
                    x=df[current_x_var].unique(),
                    y=[df[y_var][(df[current_x_var] == x) & (df[a] == k)].mean() for x in df[current_x_var].unique()],
                    marker_color=colour_list[col],
                    visible=a == k_vars[0],
                )
            )

    # Create buttons for each k_var
    buttons_k_var = []
    for a in k_vars:
        buttons_k_var.append(dict(
            method='update',
            label=a,
            args=[{
                'visible': [a == current for current in k_vars for _ in df[current].unique()],
                'title.text': f'Mean Age by {a} and {current_x_var}'
            }]
        ))

    # Create buttons for each x_var
    buttons_x_var = []
    for x in x_vars:
        buttons_x_var.append(dict(
            method='update',
            label=x,
            args=[{
                'x': [df[x].unique()] * len(k_vars) * df[k_vars[0]].nunique(),
                'y': [[df[y_var][(df[x] == val) & (df[a] == k)].mean() for val in df[x].unique()] for a in k_vars for k in df[a].unique()],
                'title.text': f'Mean Age by {k_vars[0]} and {x}'
            }]
        ))

    # Update layout with dropdowns
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=buttons_x_var,
                direction='down',
                showactive=True,
                x=-0.3,
                xanchor='left',
                y=0.9,
                yanchor='top'
            ),
            dict(
                buttons=buttons_k_var,
                direction='down',
                showactive=True,
                x=-0.3,
                xanchor='left',
                y=0.6,
                yanchor='top'
            )
        ],
        barmode='group',
        title=dict(text=title, x=0.01),
        yaxis_title='Mean Age',
        autosize=False,
        width=1100,
        height=350,
        margin=dict(l=20, r=20, t=50, b=20)
    )

    # Add annotation
    fig.update_layout(
        annotations=[
            dict(text="Select x Variable:", showarrow=False,
                x=-0.3, xref="paper", y=1, yref="paper", align="left"),
            dict(text="Select Feature:", showarrow=False,
                x=-0.3, xref="paper", y=0.7, yref="paper", align="left")
        ]
    )

    return fig


In [25]:
# Variables
x_vars = ['uACR', 'BP_Category', 'S_Ethnicity']
y_var = 'Age'
k_var = ['BMI_Category', 'BP_Category', 'S_Ethnicity', 'Ethnicity_Black', 'Gender']
colour_list = ['DarkOrange', 'Sienna', 'Chocolate', 'DarkSalmon', 'Coral', 'SandyBrown']

# Initial x_var
current_x_var = x_vars[0]

fig = go.Figure()

# Add traces for each combination of k_var values
for a in k_var:
    for col, k in enumerate(data[a].unique()):
        fig.add_trace(
            go.Bar(
                name=f'{k}',
                x=data[current_x_var].unique(),
                y=[data[y_var][(data[current_x_var] == x) & (data[a] == k)].mean() for x in data[current_x_var].unique()],
                marker_color=colour_list[col],
                visible=a == k_var[0],
            )
        )

# Create buttons for each k_var
buttons_k_var = []
for a in k_var:
    buttons_k_var.append(dict(
        method='update',
        label=a,
        args=[{
            'visible': [a == current for current in k_var for _ in data[current].unique()],
            'title.text': f'Mean Age by {a} and {current_x_var}'
        }]
    ))

# Create buttons for each x_var
buttons_x_var = []
for x in x_vars:
    buttons_x_var.append(dict(
        method='update',
        label=x,
        args=[{
            'x': [data[x].unique()] * len(k_var) * data[k_var[0]].nunique(),
            'y': [[data[y_var][(data[x] == val) & (data[a] == k)].mean() for val in data[x].unique()] for a in k_var for k in data[a].unique()],
            'title.text': f'Mean Age by {k_var[0]} and {x}'
        }]
    ))

# Update layout with dropdowns
fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons_x_var,
            direction='down',
            showactive=True,
            x=-0.3,
            xanchor='left',
            y=0.9,
            yanchor='top'
        ),
        dict(
            buttons=buttons_k_var,
            direction='down',
            showactive=True,
            x=-0.3,
            xanchor='left',
            y=0.6,
            yanchor='top'
        )
    ],
    barmode='group',
    title=dict(text='Bivariate Analysis (Age)', x=0.01),
    yaxis_title='Mean Age',
    autosize=False,
    width=1100,
    height=350,
    margin=dict(l=20, r=20, t=50, b=20)
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="Select x Variable:", showarrow=False,
             x=-0.3, xref="paper", y=1, yref="paper", align="left"),
        dict(text="Select Feature:", showarrow=False,
             x=-0.3, xref="paper", y=0.7, yref="paper", align="left")
    ]
)

fig.show()


In [26]:
# Variables
x_vars = ['uACR', 'BP_Category', 'S_Ethnicity']
y_var = 'Age'
k_var = ['BMI_Category', 'BP_Category', 'S_Ethnicity', 'Ethnicity_Black', 'Gender']
colour_list = ['DarkOrange', 'Sienna', 'Chocolate', 'DarkSalmon', 'Coral', 'SandyBrown']

# Initial x_var
current_x_var = x_vars[0]

fig = go.Figure()

# Add traces for each combination of k_var values
for a in k_var:
    for col, k in enumerate(data[a].unique()):
        fig.add_trace(
            go.Bar(
                name=f'{k}',
                x=data[current_x_var].unique(),
                y=[data[y_var][(data[current_x_var] == x) & (data[a] == k)].mean() for x in data[current_x_var].unique()],
                marker_color=colour_list[col],
                visible=a == k_var[0],
            )
        )

# Create buttons for each k_var
buttons_k_var = []
for a in k_var:
    buttons_k_var.append(dict(
        method='update',
        label=a,
        args=[{
            'visible': [a == current for current in k_var for _ in data[current].unique()],
            'title.text': f'Mean Age by {a} and {current_x_var}'
        }]
    ))

# Create buttons for each x_var
buttons_x_var = []
for x in x_vars:
    buttons_x_var.append(dict(
        method='update',
        label=x,
        args=[{
            'x': [data[x].unique()] * len(k_var) * data[k_var[0]].nunique(),
            'y': [[data[y_var][(data[x] == val) & (data[a] == k)].mean() for val in data[x].unique()] for a in k_var for k in data[a].unique()],
            'title.text': f'Mean Age by {k_var[0]} and {x}'
        }]
    ))

# Update layout with dropdowns
fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons_x_var,
            direction='down',
            showactive=True,
            x=-0.3,
            xanchor='left',
            y=0.9,
            yanchor='top'
        ),
        dict(
            buttons=buttons_k_var,
            direction='down',
            showactive=True,
            x=-0.3,
            xanchor='left',
            y=0.6,
            yanchor='top'
        )
    ],
    barmode='group',
    title=dict(text='Bivariate Analysis (Age)', x=0.01),
    yaxis_title='Mean Age',
    autosize=False,
    width=1100,
    height=350,
    margin=dict(l=20, r=20, t=50, b=20)
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="Select x Variable:", showarrow=False,
             x=-0.3, xref="paper", y=1, yref="paper", align="left"),
        dict(text="Select Feature:", showarrow=False,
             x=-0.3, xref="paper", y=0.7, yref="paper", align="left")
    ]
)

fig.show()


Chart Results
- The charts show that Black people turn out with high abnormal uACR results more than 10 years younger than non Black people.
- Black people turn out with HPT crisis results about 10 years before non black poeple.

## Statistical Tests

In [27]:
# Import modules
import statsmodels.api as sm 
from statsmodels.formula.api import ols 
import researchpy as rp

In [28]:
# Performing two-way ANOVA between Age, Ethnicity_Black and uACR
model = ols(
    'Age ~ C(Ethnicity_Black) + C(uACR) +\
    C(Ethnicity_Black):C(uACR)', data=data).fit()
sm.stats.anova_lm(model, typ=2)

,sum_sq,df,F,PR(>F)
C(Ethnicity_Black),27.977277,1.0,0.141317,0.707059
C(uACR),2941.980188,2.0,7.430153,0.000628
C(Ethnicity_Black):C(uACR),446.744571,2.0,1.128281,0.324020
Residual,189066.848542,955.0,NaN,NaN


In [29]:
# Performing two-way ANOVA between Age, Ethnicity_Black and BP_Category
model = ols(
    'Age ~ C(Ethnicity_Black) + C(BP_Category) +\
    C(Ethnicity_Black):C(BP_Category)', data=data).fit()
sm.stats.anova_lm(model, typ=2)

,sum_sq,df,F,PR(>F)
C(Ethnicity_Black),150.217829,1.0,0.783264,3.763686e-01
C(BP_Category),9728.720811,4.0,12.681839,4.653150e-10
C(Ethnicity_Black):C(BP_Category),339.792016,4.0,0.442935,7.776100e-01
Residual,182387.060474,951.0,NaN,NaN


In [30]:
import scipy.stats as stats

In [31]:
cross_eth_uacr = pd.crosstab(data['Ethnicity_Black'], data['uACR'])
stats.chi2_contingency(cross_eth_uacr)

Chi2ContingencyResult(statistic=19.024002327845235, pvalue=7.395888968551565e-05, dof=2, expected_freq=array([[ 45.80020812,   7.12174818,  65.0780437 ],
       [327.19979188,  50.87825182, 464.9219563 ]]))

In [32]:
cross_eth_bpcat = pd.crosstab(data['Ethnicity_Black'], data['BP_Category'])
stats.chi2_contingency(cross_eth_bpcat)

Chi2ContingencyResult(statistic=16.071001337530294, pvalue=0.0029253553108120017, dof=4, expected_freq=array([[ 24.18938606,   7.24453694,   1.10509886,  66.30593132,
         19.15504683],
       [172.81061394,  51.75546306,   7.89490114, 473.69406868,
        136.84495317]]))

In [33]:
crosstab, test_results, expected = rp.crosstab(data["Ethnicity_Black"], data["uACR"],
                                               test= "chi-square",
                                               expected_freqs= True,
                                               prop= "cell")

test_results

/opt/anaconda3/envs/vanilla/lib/python3.12/site-packages/researchpy/crosstab.py:146: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Pearson Chi-square ( 2.0) = ' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.



,Chi-square test,results
0,Pearson Chi-square ( 2.0) =,19.0240
1,p-value =,0.0001
2,Cramer's V =,0.1407


In [34]:
crosstab, test_results, expected = rp.crosstab(data["Ethnicity_Black"], data["BP_Category"],
                                               test= "chi-square",
                                               expected_freqs= True,
                                               prop= "cell")

test_results

/opt/anaconda3/envs/vanilla/lib/python3.12/site-packages/researchpy/crosstab.py:146: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Pearson Chi-square ( 4.0) = ' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.



,Chi-square test,results
0,Pearson Chi-square ( 4.0) =,16.0710
1,p-value =,0.0029
2,Cramer's V =,0.1293


In [35]:
data['Pulse Pressure'] = data['Systolic'] - data['Diastolic']

# Variables
x_vars = ['uACR', 'BMI_Category', 'BP_Category', 'Ethnicity_Black', 'Gender']
y_var = 'Pulse Pressure'
k_var = ['BMI_Category', 'BP_Category', 'S_Ethnicity', 'Ethnicity_Black', 'Gender', 'uACR']
colour_list = ['DarkOrange', 'Sienna', 'Chocolate', 'DarkSalmon', 'Coral', 'SandyBrown']

# Initial x_var
current_x_var = x_vars[0]

fig = go.Figure()

# Add traces for each combination of k_var values
for a in k_var:
    for col, k in enumerate(data[a].unique()):
        fig.add_trace(
            go.Bar(
                name=f'{k}',
                x=data[current_x_var].unique(),
                y=[data[y_var][(data[current_x_var] == x) & (data[a] == k)].mean() for x in data[current_x_var].unique()],
                marker_color=colour_list[col],
                visible=a == k_var[0],
            )
        )

# Create buttons for each k_var
buttons_k_var = []
for a in k_var:
    buttons_k_var.append(dict(
        method='update',
        label=a,
        args=[{
            'visible': [a == current for current in k_var for _ in data[current].unique()],
            'title.text': f'Mean Age by {a} and {current_x_var}'
        }]
    ))

# Create buttons for each x_var
buttons_x_var = []
for x in x_vars:
    buttons_x_var.append(dict(
        method='update',
        label=x,
        args=[{
            'x': [data[x].unique()] * len(k_var) * data[k_var[0]].nunique(),
            'y': [[data[y_var][(data[x] == val) & (data[a] == k)].mean() for val in data[x].unique()] for a in k_var for k in data[a].unique()],
            'title.text': f'Mean Pulse Pressure by {k_var[0]} and {x}'
        }]
    ))

# Update layout with dropdowns
fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons_x_var,
            direction='down',
            showactive=True,
            x=-0.3,
            xanchor='left',
            y=0.9,
            yanchor='top'
        ),
        dict(
            buttons=buttons_k_var,
            direction='down',
            showactive=True,
            x=-0.3,
            xanchor='left',
            y=0.6,
            yanchor='top'
        )
    ],
    barmode='group',
    title=dict(text='Bivariate Analysis (Pulse Pressure)', x=0.01),
    yaxis_title='Mean Pulse Pressure',
    autosize=False,
    width=1100,
    height=350,
    margin=dict(l=20, r=20, t=50, b=20)
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="Select x Variable:", showarrow=False,
             x=-0.3, xref="paper", y=1, yref="paper", align="left"),
        dict(text="Select Feature:", showarrow=False,
             x=-0.3, xref="paper", y=0.7, yref="paper", align="left")
    ]
)

fig.show()


Chart Results
- Mean pulse pressure for Black people with high abnormal uACR results is over 20 mmHg lower than non Black participants.
- There is a sharper gradient for mean pulse pressure across BMI categories (from 44.5 to 56.8) among non Black participants than among Black participants (52.0 to 55.1). This could be due to the deficiencies of BMI measures of Black people.